# Fire 2021 Interactive Prediction Notebook

This notebook does the following:
- Loads one fire from year 2021 with 10 leading observations
- Displays 10-day observed timeline on Google Maps basemap with day slider
- Extracts base feature rasters from Google Earth Engine
- Allows user to scale GEE features (0.5–2.0x) to explore prediction sensitivity
- Updates next-day fire spread prediction in real-time

In [16]:
import os
import sys
import base64
from io import BytesIO
from pathlib import Path
from dataclasses import replace

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import rasterio
from rasterio.enums import Resampling

import ipywidgets as widgets
from IPython.display import display, clear_output
from ipyleaflet import Map, TileLayer, ImageOverlay, LayersControl, WidgetControl

REPO_ROOT = Path('/u50/overallt/WildfireSpreadPrediction')
BACKEND_SRC = REPO_ROOT / 'backend' / 'src'
WSTS_SRC = REPO_ROOT / 'src' / 'wsts' / 'src'
FEATURE_API = REPO_ROOT / 'featureAPI'
EE_PROJECT = "ee-neeljos24"

if str(BACKEND_SRC) not in sys.path:
    sys.path.insert(0, str(BACKEND_SRC))
if str(WSTS_SRC) not in sys.path:
    sys.path.insert(0, str(WSTS_SRC))
if str(FEATURE_API) not in sys.path:
    sys.path.insert(0, str(FEATURE_API))

from wildfire_api.config import Settings
from wildfire_api.repository import WildfireRepository
from wildfire_api.preprocessing import SamplePreprocessor
from models import DomainAdversarialUTAELightning, ResNet18UTAELightning, UTAELightning, SMPModel

try:
    from gee_feature_extractor import GEEFeatureExtractor, normalize_features
except ImportError:
    print('Warning: gee_feature_extractor not found (GEE features will be unavailable)')
    GEEFeatureExtractor = None

torch.set_grad_enabled(False)
print('Imports ready')
print(f'EE project: {EE_PROJECT or "(not set)"}')

Imports ready
EE project: ee-neeljos24


## GEE Authentication
Credentials are already authenticated.
Run the next cell to initialize Earth Engine for this notebook session.

In [17]:
# Google Earth Engine initialization (project is required)
import ee

if not EE_PROJECT:
    raise ValueError(
        "EE_PROJECT is not set. In terminal run:\n"
        "export EE_PROJECT=<your-gcp-project-id>\n"
        "Then restart kernel and run from Cell 2."
    )
ee.Authenticate()
ee.Initialize(project="ee-neeljos24", opt_url='https://earthengine-highvolume.googleapis.com')
print(f'✓ GEE initialized with project: {EE_PROJECT}')

✓ GEE initialized with project: ee-neeljos24


## Configuration

In [18]:
CHECKPOINT_PATH = REPO_ROOT / 'featureAPI' / 'model.ckpt'
HDF5_ROOT = Path('/u50/capstone/cs4zp6g17/data/hdf5')

settings = Settings(
    model_path=CHECKPOINT_PATH,
    hdf5_root=HDF5_ROOT,
    stats_years=(2018, 2019),
    default_year=2021,
    n_leading_observations=10,
    probability_threshold=0.5,
    flatten_temporal_dimension=False,
)

repo = WildfireRepository(settings)
preprocessor = SamplePreprocessor(settings)
catalog_2021 = repo.list_year(2021)
print(f'2021 fires available: {len(catalog_2021)}')
print('(GEE should be authenticated by now — see cell above)')

2021 fires available: 156
(GEE should be authenticated by now — see cell above)


In [19]:
MODEL_CLASS_MAP = {
    'DomainAdversarialUTAELightning': DomainAdversarialUTAELightning,
    'ResNet18UTAELightning': ResNet18UTAELightning,
    'UTAELightning': UTAELightning,
    'SMPModel': SMPModel,
}

MODEL_CLASS_NAME = 'DomainAdversarialUTAELightning'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ModelClass = MODEL_CLASS_MAP[MODEL_CLASS_NAME]
model = ModelClass.load_from_checkpoint(str(CHECKPOINT_PATH), map_location=device)
model.eval()
model.to(device)
print(f'Model loaded: {MODEL_CLASS_NAME} on {device}')

Model loaded: DomainAdversarialUTAELightning on cuda


## Load fire from 2021 catalog and extract GEE base rasters

In [20]:
def array_to_data_url(arr, cmap='hot', alpha=0.65, vmin=None, vmax=None):
    arr = np.asarray(arr, dtype=np.float32)
    if vmin is not None and vmax is not None and vmax > vmin:
        norm = np.clip((arr - float(vmin)) / (float(vmax) - float(vmin) + 1e-8), 0.0, 1.0)
    elif np.nanmax(arr) > np.nanmin(arr):
        norm = (arr - np.nanmin(arr)) / (np.nanmax(arr) - np.nanmin(arr) + 1e-8)
    else:
        norm = np.zeros_like(arr)

    cm = plt.get_cmap(cmap)
    rgba = cm(norm)
    rgba[..., 3] = alpha * (norm > 0).astype(np.float32)
    img = (rgba * 255).astype(np.uint8)

    pil_img = Image.fromarray(img)
    buf = BytesIO()
    pil_img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    return f'data:image/png;base64,{b64}'

def bounds_from_center(lat, lon, height, width, meters_per_pixel=375):
    lat_deg_per_meter = 1 / 111320
    lon_deg_per_meter = 1 / (111320 * np.cos(np.deg2rad(lat)) + 1e-8)
    half_h_m = 0.5 * height * meters_per_pixel
    half_w_m = 0.5 * width * meters_per_pixel
    south = lat - half_h_m * lat_deg_per_meter
    north = lat + half_h_m * lat_deg_per_meter
    west = lon - half_w_m * lon_deg_per_meter
    east = lon + half_w_m * lon_deg_per_meter
    return ((south, west), (north, east))

# Select largest fire from 2021
size_rank = sorted(catalog_2021, key=lambda m: (m.height * m.width * max(m.samples, 1)), reverse=True)
LARGE_FIRE_RANK = 0
MANUAL_FIRE_ID = None

fire_id = MANUAL_FIRE_ID if MANUAL_FIRE_ID else size_rank[LARGE_FIRE_RANK].fire_id

cube_obj = repo.load_cube(fire_id=fire_id, year=2021)
cube = cube_obj.cube
meta = cube_obj.metadata

if cube.shape[0] < 11:
    raise RuntimeError('Selected fire has fewer than 11 frames')

sample_offset = min(0, cube.shape[0] - 11)
prep = preprocessor.prepare(cube, sample_offset=sample_offset)

# Initial prediction
x_input = prep.tensor.to(device)
with torch.inference_mode():
    y_pred = model(x_input)
    y_prob = torch.sigmoid(y_pred).detach().cpu().numpy()[0, 0]

start = prep.sample_index
obs_10day = cube[start:start+10, -1]

crop_h, crop_w = prep.spatial_shape
orig_h, orig_w = obs_10day.shape[-2:]
h_off = (orig_h - crop_h) // 2
w_off = (orig_w - crop_w) // 2
obs_10day = obs_10day[:, h_off:h_off+crop_h, w_off:w_off+crop_w]

print(f'Fire: {fire_id} (rank {LARGE_FIRE_RANK + 1}/{len(size_rank)})')
print(f'Cube, model input: {cube.shape} → {tuple(prep.tensor.shape)}')
print(f'Observed timeline: {obs_10day.shape}, prediction: {y_prob.shape}')

# Extract base rasters from GEE
print('\nExtracting base rasters from GEE...')
base_rasters = None
try:
    if GEEFeatureExtractor is not None:
        gee_extractor = GEEFeatureExtractor(project=EE_PROJECT)
        fire_region_size = max(crop_h, crop_w) * 375 / 1000
        gee_features = gee_extractor.extract_features(
            latitude=meta.latitude,
            longitude=meta.longitude,
            date='2021-06-01',
            region_size_meters=int(fire_region_size * 1000),
            resolution_meters=375
        )
        if isinstance(gee_features, tuple):
            gee_features = gee_features[0]
        base_rasters = {
            'viirs_m11': gee_features[0],
            'viirs_i2': gee_features[1],
            'ndvi': gee_features[2],
            'evi2': gee_features[3],
            'precip': gee_features[4],
            'wind_speed': gee_features[5],
            'elevation': gee_features[6],
        }
        print(f'✓ Extracted {len(base_rasters)} features from GEE')
        for key, arr in base_rasters.items():
            print(f'  {key}: min={np.nanmin(arr):.2f}, max={np.nanmax(arr):.2f}')
    else:
        print('GEEFeatureExtractor not available')
except Exception as e:
    print(f'GEE extraction failed: {e}')

# Fallback if GEE returns invalid/all-zero data so sliders still drive prediction changes
if base_rasters is None:
    base_rasters = {}

variable_keys = ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'wind_speed']
invalid_keys = []
for k in variable_keys:
    arr = base_rasters.get(k)
    if arr is None or not np.isfinite(arr).any() or np.nanmax(np.abs(arr)) < 1e-8:
        invalid_keys.append(k)

if invalid_keys:
    print(f'Using HDF5 fallback for: {", ".join(invalid_keys)}')
    fallback_map = {
        'viirs_m11': 0,
        'viirs_i2': 1,
        'ndvi': 3,
        'evi2': 4,
        'precip': 5,
        'wind_speed': 6,
    }
    last_window = cube[start + 9, :, h_off:h_off+crop_h, w_off:w_off+crop_w]
    for k in invalid_keys:
        base_rasters[k] = np.asarray(last_window[fallback_map[k]], dtype=np.float32)

Fire: fire_25294714 (rank 1/156)
Cube, model input: (72, 23, 302, 235) → (1, 10, 40, 288, 224)
Observed timeline: (10, 288, 224), prediction: (288, 224)

Extracting base rasters from GEE...
✓ Extracted 7 features from GEE
  viirs_m11: min=0.00, max=0.00
  viirs_i2: min=0.00, max=0.00
  ndvi: min=0.00, max=0.00
  evi2: min=0.00, max=0.00
  precip: min=0.00, max=0.00
  wind_speed: min=0.00, max=0.00
  elevation: min=0.00, max=0.00
Using HDF5 fallback for: viirs_m11, viirs_i2, ndvi, evi2, precip, wind_speed


## Interactive map with day slider

In [21]:
google_sat = TileLayer(url='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', attribution='Google', name='Google Satellite')

center = (meta.latitude, meta.longitude)
bounds = bounds_from_center(meta.latitude, meta.longitude, crop_h, crop_w, meters_per_pixel=375)

overlay_day = ImageOverlay(url=array_to_data_url(obs_10day[0], cmap='hot', alpha=0.7), bounds=bounds, name='Observed fire')
overlay_pred = ImageOverlay(url=array_to_data_url(y_prob, cmap='viridis', alpha=0.6, vmin=0.0, vmax=1.0), bounds=bounds, name='Predicted fire')

m = Map(center=center, zoom=10, basemap=google_sat)
m.add_layer(overlay_day)
m.add_layer(overlay_pred)
m.add_control(LayersControl(position='topright'))
m.fit_bounds(bounds)

day_slider = widgets.IntSlider(value=0, min=0, max=9, step=1, description='Day', layout=widgets.Layout(width='200px'))
day_label = widgets.HTML(value='<b>Day:</b> 1 / 10')
zoom_slider = widgets.IntSlider(value=10, min=5, max=16, step=1, description='Zoom', layout=widgets.Layout(width='200px'))

def on_day_change(change):
    if change['name'] == 'value':
        idx = change['new']
        overlay_day.url = array_to_data_url(obs_10day[idx], cmap='hot', alpha=0.7)
        day_label.value = f'<b>Day:</b> {idx+1} / 10'

def on_zoom_change(change):
    if change['name'] == 'value':
        m.zoom = change['new']

day_slider.observe(on_day_change, names='value')
zoom_slider.observe(on_zoom_change, names='value')

slider_box = widgets.VBox([day_label, day_slider, zoom_slider])
m.add_control(WidgetControl(widget=slider_box, position='bottomleft'))

print('Map prepared. Displayed with scale sliders in the next cell.')

Map prepared. Displayed with scale sliders in the next cell.


## Scale feature inputs from GEE base rasters
Adjust 0.5–2.0 scale factors to explore fire spread sensitivity.

In [25]:
scale_widgets = {
    'viirs_m11': widgets.FloatSlider(description='M11', value=1.0, min=0.5, max=2.0, step=0.1, layout=widgets.Layout(width='48%')),
    'viirs_i2': widgets.FloatSlider(description='I2', value=1.0, min=0.5, max=2.0, step=0.1, layout=widgets.Layout(width='48%')),
    'ndvi': widgets.FloatSlider(description='NDVI', value=1.0, min=0.5, max=2.0, step=0.1, layout=widgets.Layout(width='48%')),
    'evi2': widgets.FloatSlider(description='EVI2', value=1.0, min=0.5, max=2.0, step=0.1, layout=widgets.Layout(width='48%')),
    'precip': widgets.FloatSlider(description='Precip', value=1.0, min=0.5, max=2.0, step=0.1, layout=widgets.Layout(width='48%')),
    'wind_speed': widgets.FloatSlider(description='Wind', value=1.0, min=0.5, max=2.0, step=0.1, layout=widgets.Layout(width='48%')),
}

# Static terrain features from HDF5
elev_mean = float(np.nanmean(cube[start:start+10, 14, h_off:h_off+crop_h, w_off:w_off+crop_w]))
slope_mean = float(np.nanmean(cube[start:start+10, 12, h_off:h_off+crop_h, w_off:w_off+crop_w]))
aspect_mean = float(np.nanmean(cube[start:start+10, 13, h_off:h_off+crop_h, w_off:w_off+crop_w]))

static_info = widgets.VBox([
    widgets.HTML(value=f'<b>Terrain (fixed):</b> Elev={elev_mean:.1f}, Slope={slope_mean:.2f}, Aspect={aspect_mean:.1f}'),
])

# Layout
rows = []
scale_keys = list(scale_widgets.keys())
for i in range(0, len(scale_keys), 2):
    pair = [scale_widgets[scale_keys[i]]]
    if i + 1 < len(scale_keys):
        pair.append(scale_widgets[scale_keys[i+1]])
    rows.append(widgets.HBox(pair))

export_checkbox = widgets.Checkbox(value=False, description='Export scaled rasters to GeoTIFF')
export_dir = widgets.Text(description='Export dir', value=str(REPO_ROOT / 'featureAPI' / 'generated_rasters'), layout=widgets.Layout(width='95%'))

status_out = widgets.Output()
heatmap_out = widgets.Output()

controls_panel = widgets.VBox([
    widgets.HTML(value='<b>Scale factors (0.5=half, 1.0=original, 2.0=double):</b>'),
    *rows,
    static_info,
    export_checkbox,
    export_dir,
    widgets.HTML(value='<i>Prediction auto-updates when a scale slider changes.</i>'),
    status_out,
    heatmap_out
])

display(widgets.VBox([m, controls_panel]))

In [26]:
RAW_CHANNEL_MAP = {
    'viirs_m11': 0, 'viirs_i2': 1, 'ndvi': 3, 'evi2': 4, 'precip': 5,
    'wind_speed': 6, 'elevation': 14, 'slope': 12, 'aspect': 13,
}

def scale_raster(base_arr, factor):
    return np.asarray(base_arr, dtype=np.float32) * float(factor)

def resize_to_shape(arr, target_shape):
    """Nearest-neighbor resize for 2D arrays without extra deps."""
    arr = np.asarray(arr, dtype=np.float32)
    th, tw = target_shape
    h, w = arr.shape
    if (h, w) == (th, tw):
        return arr

    y_idx = np.linspace(0, h - 1, th).astype(np.int32)
    x_idx = np.linspace(0, w - 1, tw).astype(np.int32)
    return arr[np.ix_(y_idx, x_idx)].astype(np.float32)

def write_tif(path, arr, lat, lon, resolution=375):
    from rasterio.transform import from_bounds
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    h, w = arr.shape
    b = bounds_from_center(lat, lon, h, w, meters_per_pixel=resolution)
    south, west = b[0]
    north, east = b[1]
    xform = from_bounds(west, south, east, north, w, h)
    with rasterio.open(path, 'w', driver='GTiff', height=h, width=w, count=1, dtype='float32', crs='EPSG:4326', transform=xform) as dst:
        dst.write(arr.astype(np.float32), 1)

def update_prediction(_=None):
    with status_out:
        clear_output()
        if base_rasters is None:
            print('No GEE base rasters loaded')
            return
        print('Applying scale factors...')

    with heatmap_out:
        clear_output()

    raw_window = np.copy(cube[start:start+10, :, h_off:h_off+crop_h, w_off:w_off+crop_w])

    try:
        generated = {}

        # Scale variable features and ensure spatial shape matches model crop
        for key in ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'wind_speed']:
            if key in base_rasters:
                scale = float(scale_widgets[key].value)
                scaled = scale_raster(base_rasters[key], scale)
                scaled = resize_to_shape(scaled, (crop_h, crop_w))
                generated[key] = scaled
            else:
                print(f'{key} not in base_rasters')
                return

        # Fixed terrain from HDF5
        generated['elevation'] = raw_window[-1, 14].astype(np.float32)
        generated['slope'] = raw_window[-1, 12].astype(np.float32)
        generated['aspect'] = raw_window[-1, 13].astype(np.float32)

        # Update cube
        for key, ch in RAW_CHANNEL_MAP.items():
            if key in generated:
                raw_window[-1, ch] = generated[key]

        temp_cube = np.copy(cube)
        temp_cube[start:start+10, :, h_off:h_off+crop_h, w_off:w_off+crop_w] = raw_window

        prep_user = preprocessor.prepare(temp_cube, sample_offset=sample_offset)
        x_user = prep_user.tensor.to(device)

        with torch.inference_mode():
            y_user = model(x_user)
            y_user_prob = torch.sigmoid(y_user).detach().cpu().numpy()[0, 0]

        overlay_pred.url = array_to_data_url(y_user_prob, cmap='viridis', alpha=0.65, vmin=0.0, vmax=1.0)

        exported = []
        if export_checkbox.value:
            exp_dir = Path(export_dir.value).expanduser().resolve()
            exp_dir.mkdir(parents=True, exist_ok=True)
            for key, arr in generated.items():
                out_path = exp_dir / f'{fire_id}_{key}.tif'
                write_tif(out_path, arr, meta.latitude, meta.longitude)
                exported.append(str(out_path))

        with status_out:
            clear_output()
            scales = ', '.join([f'{k}={float(scale_widgets[k].value):.1f}x' for k in ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'wind_speed']])
            delta_mean = float(np.mean(np.abs(y_user_prob - y_prob)))
            print(f'✓ Prediction updated\nScales: {scales}')
            print(f'Prediction stats: min={float(np.min(y_user_prob)):.4f}, max={float(np.max(y_user_prob)):.4f}, mean={float(np.mean(y_user_prob)):.4f}')
            print(f'Δ vs baseline mean|abs|={delta_mean:.6f}')
            print(f'Raster size used: {crop_h}x{crop_w}')
            if exported:
                print(f'\nExported {len(exported)} TIFFs')

        with heatmap_out:
            keys = ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'elevation', 'slope', 'aspect', 'wind_speed']
            fig, axes = plt.subplots(3, 3, figsize=(12, 10))
            axes = axes.flatten()
            for idx, key in enumerate(keys):
                arr = generated[key]
                cmap_use = 'viridis' if key == 'wind_speed' else 'magma'
                im = axes[idx].imshow(arr, cmap=cmap_use)
                axes[idx].set_title(key, fontsize=9)
                plt.colorbar(im, ax=axes[idx], fraction=0.046)
            plt.tight_layout()
            plt.show()

    except Exception as e:
        with status_out:
            clear_output()
            print(f'Update failed: {e}')
            import traceback
            traceback.print_exc()

for key in ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'wind_speed']:
    scale_widgets[key].observe(update_prediction, names='value')

print('Auto-update handlers ready')

Auto-update handlers ready


In [27]:
# Clear stale slider observers, then bind exactly one auto-update observer per scale slider
for key in ['viirs_m11', 'viirs_i2', 'ndvi', 'evi2', 'precip', 'wind_speed']:
    try:
        scale_widgets[key]._trait_notifiers['value']['change'].clear()
    except Exception:
        pass
    scale_widgets[key].observe(update_prediction, names='value')

# Run once so the current slider state is reflected on the map
update_prediction()
print('✓ Auto-update observers rebound and initial prediction rendered')

✓ Auto-update observers rebound and initial prediction rendered


## How it works
1. **Load**: Fire loaded from HDF5 (2021 catalog, largest by area)
2. **Extract**: Base rasters extracted from GEE (VIIRS, MODIS, CHIRPS, ERA5)
3. **Scale**: User adjusts 6 sliders (0.5–2.0x) to modulate GEE features
4. **Predict**: Click submit → scales applied → model inference → map updates with next-day prediction
5. **Export** (optional): Scaled rasters saved as GeoTIFF

**Terrain features** (elevation, slope, aspect) are **fixed** from the fire's HDF5 cube and not scaled.